In [ ]:
import torch
import torchvision
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
import math
from pathlib import Path
from PIL import Image
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
import random
import numpy as np
import timm
from tqdm.auto import tqdm
import copy
import gc
from torch.amp import autocast, GradScaler
from collections import deque


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

In [ ]:
class NICOPPDataset(Dataset):
    def __init__(
        self,
        root_path,
        domain_groups,
        splits=("train",),
        real_domains=None,
        class_to_idx=None,
        transform=None
    ):
        self.root_path = Path(root_path)
        self.domain_groups = domain_groups if isinstance(domain_groups, (list, tuple)) else [domain_groups]
        self.splits = splits if isinstance(splits, (list, tuple)) else [splits]
        self.real_domains = set(real_domains) if real_domains is not None else None
        self.transform = transform
        self.samples = []

        # Build class mapping
        if class_to_idx is None:
            class_names = set()

            for domain_group in self.domain_groups:
                for split in self.splits:
                    split_path = self.root_path / domain_group / split
                    if not split_path.exists():
                        continue

                    for real_domain_dir in split_path.iterdir():
                        if not real_domain_dir.is_dir():
                            continue

                        if self.real_domains is not None and real_domain_dir.name not in self.real_domains:
                            continue

                        for class_dir in real_domain_dir.iterdir():
                            if class_dir.is_dir():
                                class_names.add(class_dir.name)

            self.class_to_idx = {
                class_name: idx
                for idx, class_name in enumerate(sorted(class_names))
            }
        else:
            self.class_to_idx = class_to_idx

        # Collect samples
        for domain_group in self.domain_groups:
            for split in self.splits:
                split_path = self.root_path / domain_group / split
                if not split_path.exists():
                    continue

                for real_domain_dir in split_path.iterdir():
                    if not real_domain_dir.is_dir():
                        continue

                    real_domain = real_domain_dir.name

                    if self.real_domains is not None and real_domain not in self.real_domains:
                        continue

                    for class_dir in real_domain_dir.iterdir():
                        if not class_dir.is_dir():
                            continue

                        class_name = class_dir.name

                        if class_name not in self.class_to_idx:
                            continue

                        label = self.class_to_idx[class_name]

                        for img_path in class_dir.rglob("*"):
                            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                                self.samples.append({
                                    "image_path": img_path,
                                    "label": label,
                                    "class_name": class_name,
                                    "domain_group": domain_group,
                                    "real_domain": real_domain,
                                    "split": split,
                                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = sample["label"]
        return image, label

In [ ]:
def create_model(model_path, num_classes, pretrained=False):
    model = timm.create_model(model_path, pretrained=pretrained)
    model.classifier = nn.Linear(
        in_features=model.classifier.in_features,
        out_features=num_classes,
        bias=True
    )
    return model


In [ ]:
class AveragedModel(nn.Module):
    """
    Standalone adaptation of DomainBed-v2/domainbed/lib/swa_utils.py AveragedModel.

    Important: this averages PARAMETERS only, matching DomainBed-v2's implementation.
    BatchNorm buffers are refreshed later with _update_bn_statistics() when training from scratch.
    """
    def __init__(self, model, device=None):
        super().__init__()
        self.start_step = -1
        self.end_step = -1

        if isinstance(model, AveragedModel):
            model = model.module

        self.module = copy.deepcopy(model)

        if device is not None:
            self.module = self.module.to(device)

        self.register_buffer(
            "n_averaged",
            torch.tensor(0, dtype=torch.long, device=device)
        )

    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)

    def predict(self, *args, **kwargs):
        return self.module(*args, **kwargs)

    def update_parameters(self, model, step=None, start_step=None, end_step=None):
        if isinstance(model, AveragedModel):
            model = model.module

        for p_swa, p_model in zip(self.module.parameters(), model.parameters()):
            device = p_swa.device
            p_model_ = p_model.detach().to(device)

            if self.n_averaged == 0:
                p_swa.detach().copy_(p_model_)
            else:
                p_swa.detach().copy_(
                    p_swa.detach()
                    + (p_model_ - p_swa.detach()) / (self.n_averaged.to(device) + 1)
                )

        self.n_averaged += 1

        if step is not None:
            if start_step is None:
                start_step = step
            if end_step is None:
                end_step = step

        if start_step is not None:
            if self.n_averaged == 1:
                self.start_step = start_step

        if end_step is not None:
            self.end_step = end_step


class LossValley:
    """
    Standalone adaptation of DomainBed-v2/domainbed/lib/swad.py LossValley.

    DomainBed-v2 SWAD does NOT choose one best checkpoint. It:
      1. densely averages weights inside each validation segment,
      2. detects convergence from a source-validation-loss valley,
      3. averages accepted segment-SWA models,
      4. stops when the loss valley is dead.
    """
    def __init__(self, n_converge=3, n_tolerance=6, tolerance_ratio=0.3):
        self.n_converge = n_converge
        self.n_tolerance = n_tolerance
        self.tolerance_ratio = tolerance_ratio

        self.converge_Q = deque(maxlen=n_converge)
        self.smooth_Q = deque(maxlen=n_tolerance)

        self.final_model = None
        self.converge_step = None
        self.dead_valley = False
        self.threshold = None

    @property
    def is_converged(self):
        return self.converge_step is not None

    def get_smooth_loss(self, idx):
        return min([model.end_loss for model in list(self.smooth_Q)[idx:]])

    def update_and_evaluate(self, segment_swa, val_acc, val_loss):
        # val_acc is kept for API similarity with DomainBed-v2, but LossValley uses val_loss.
        if self.dead_valley:
            return

        frozen = copy.deepcopy(segment_swa).cpu()
        frozen.end_loss = float(val_loss)

        self.converge_Q.append(frozen)
        self.smooth_Q.append(frozen)

        # Before convergence: wait until the best loss in the convergence window
        # becomes the oldest segment. This follows DomainBed-v2 exactly.
        if not self.is_converged:
            if len(self.converge_Q) < self.n_converge:
                return

            losses = [model.end_loss for model in self.converge_Q]
            min_idx = int(np.argmin(losses))
            untilmin_segment_swa = self.converge_Q[min_idx]

            if min_idx == 0:
                self.converge_step = self.converge_Q[0].end_step
                self.final_model = AveragedModel(untilmin_segment_swa)

                th_base = float(np.mean(losses))
                self.threshold = th_base * (1.0 + self.tolerance_ratio)

                if self.n_tolerance < self.n_converge:
                    for i in range(self.n_converge - self.n_tolerance):
                        model = self.converge_Q[1 + i]
                        self.final_model.update_parameters(
                            model,
                            start_step=model.start_step,
                            end_step=model.end_step,
                        )

                elif self.n_tolerance > self.n_converge:
                    converge_idx = self.n_tolerance - self.n_converge
                    Q = list(self.smooth_Q)[: converge_idx + 1]
                    start_idx = 0

                    for i in reversed(range(len(Q))):
                        model = Q[i]
                        if model.end_loss > self.threshold:
                            start_idx = i + 1
                            break

                    for model in Q[start_idx + 1:]:
                        self.final_model.update_parameters(
                            model,
                            start_step=model.start_step,
                            end_step=model.end_step,
                        )

                print(
                    f"Model converged at step {self.converge_step}, "
                    f"Start step = {self.final_model.start_step}; "
                    f"Threshold = {self.threshold:.6f}"
                )

            return

        # After convergence: accept old segment-SWA models while the smoothed
        # validation loss is still inside the loss valley.
        if self.smooth_Q[0].end_step < self.converge_step:
            return

        min_vloss = self.get_smooth_loss(0)

        if min_vloss > self.threshold:
            self.dead_valley = True
            print(f"Valley is dead at step {self.final_model.end_step}")
            return

        model = self.smooth_Q[0]
        self.final_model.update_parameters(
            model,
            start_step=model.start_step,
            end_step=model.end_step,
        )

    def get_final_model(self, device="cuda"):
        if not self.is_converged:
            print("Requested final model, but model is not yet converged; returning last segment SWA instead")
            return self.converge_Q[-1].to(device)

        # Flush the remaining tolerated queue exactly like DomainBed-v2.
        if not self.dead_valley:
            if len(self.smooth_Q) > 0:
                self.smooth_Q.popleft()

            while len(self.smooth_Q) > 0:
                smooth_loss = self.get_smooth_loss(0)

                if smooth_loss > self.threshold:
                    break

                segment_swa = self.smooth_Q.popleft()
                self.final_model.update_parameters(
                    segment_swa,
                    step=segment_swa.end_step,
                )

        return self.final_model.to(device)


class Trainer:
    def __init__(
        self,
        model_path: str = None,
        num_classes: int = None,
        pretrained: bool = False,
        seed: int = 42,
        root_path: str = "/kaggle/input/datasets/trieuvuongnguyen/nicopp-rethink-splitted/NICO++",
        target_domain=None,
        source_transform=None,
        target_transform=None,
        batch_size: int = 32,
        lr: float = 1e-4,
        weight_decay: float = 1e-4,
        max_iters: int = 10000,
        val_ratio: float = 0.2,
        source_splits=("train",),
        target_splits=("test",),
        num_workers: int = 0,
        use_amp: bool = True,
        use_data_parallel: bool = True,
        use_swad: bool = True,
        swad_n_converge: int = 3,
        swad_n_tolerance: int = 6,
        swad_tolerance_ratio: float = 0.3,
        swad_update_bn: bool = True,
        swad_bn_steps: int = 500,
        swad_early_stop: bool = True,
    ):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.seed = seed
        set_seed(self.seed)

        self.root_path = root_path
        self.target_domain = target_domain
        self.target_domain_groups = (
            list(target_domain)
            if isinstance(target_domain, (list, tuple))
            else [target_domain]
        )

        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.max_iters = max_iters

        self.val_ratio = val_ratio
        self.source_splits = tuple(source_splits)
        self.target_splits = tuple(target_splits)

        self.num_workers = num_workers
        self.use_amp = use_amp and torch.cuda.is_available()
        self.use_data_parallel = use_data_parallel
        self.scaler = GradScaler("cuda", enabled=self.use_amp)
        self.pretrained = pretrained

        # SWAD settings from DomainBed-v2 defaults:
        # n_converge=3, n_tolerance=6, tolerance_ratio=0.3
        self.use_swad = use_swad
        self.swad_n_converge = swad_n_converge
        self.swad_n_tolerance = swad_n_tolerance
        self.swad_tolerance_ratio = swad_tolerance_ratio
        self.swad_update_bn = swad_update_bn
        self.swad_bn_steps = swad_bn_steps
        self.swad_early_stop = swad_early_stop

        self.swad = None
        self.swad_segment_swa = None
        self.swad_final_model = None
        self.swad_state_dict = None

        all_domain_groups = sorted([
            d.name for d in Path(root_path).iterdir()
            if d.is_dir()
        ])

        if target_domain is None:
            raise ValueError("target_domain must be specified.")

        missing_targets = [d for d in self.target_domain_groups if d not in all_domain_groups]
        if len(missing_targets) > 0:
            raise ValueError(
                f"target_domain {missing_targets} not found. "
                f"Available domain groups: {all_domain_groups}"
            )

        self.source_domain_groups = [
            d for d in all_domain_groups
            if d not in self.target_domain_groups
        ]

        print(f"Target group(s): {self.target_domain_groups}")
        print(f"Source groups: {self.source_domain_groups}")

        # Build one global class mapping. This reads class-folder names only.
        class_index_dataset = NICOPPDataset(
            root_path=self.root_path,
            domain_groups=all_domain_groups,
            splits=source_splits,
            transform=None,
        )
        self.class_to_idx = class_index_dataset.class_to_idx
        print(f"Number of classes: {len(self.class_to_idx)}")

        if num_classes is None:
            num_classes = len(self.class_to_idx)

        model = create_model(
            model_path=model_path,
            num_classes=num_classes,
            pretrained=pretrained,
        )
        model = model.to(self.device)

        if self.use_data_parallel and torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
            model = nn.DataParallel(model)

        self.model = model

        # Build one train loader per real source domain, matching DomainBed's
        # multi-domain minibatch idea.
        self.source_train_loaders = []
        self.source_val_loaders = []
        self.source_val_subsets = []
        self.source_domain_names = []

        domain_counter = 0

        for domain_group in self.source_domain_groups:
            real_domains = self._get_real_domains(domain_group, source_splits)

            for real_domain in real_domains:
                train_dataset_full = NICOPPDataset(
                    root_path=self.root_path,
                    domain_groups=[domain_group],
                    splits=source_splits,
                    real_domains=[real_domain],
                    class_to_idx=self.class_to_idx,
                    transform=source_transform,
                )

                val_dataset_full = NICOPPDataset(
                    root_path=self.root_path,
                    domain_groups=[domain_group],
                    splits=source_splits,
                    real_domains=[real_domain],
                    class_to_idx=self.class_to_idx,
                    transform=target_transform,
                )

                n = len(train_dataset_full)
                if n == 0:
                    raise ValueError(
                        f"Empty source domain: group={domain_group}, real_domain={real_domain}"
                    )

                val_size = int(n * self.val_ratio)
                train_size = n - val_size

                generator = torch.Generator().manual_seed(self.seed + domain_counter)
                indices = torch.randperm(n, generator=generator).tolist()

                train_indices = indices[:train_size]
                val_indices = indices[train_size:]

                train_subset = Subset(train_dataset_full, train_indices)
                val_subset = Subset(val_dataset_full, val_indices)

                loader_kwargs = dict(
                    batch_size=self.batch_size,
                    shuffle=True,
                    num_workers=self.num_workers,
                    pin_memory=True,
                    drop_last=True,
                )

                if self.num_workers > 0:
                    loader_kwargs.update(
                        persistent_workers=True,
                        prefetch_factor=2,
                    )

                train_loader = DataLoader(train_subset, **loader_kwargs)
                val_loader = DataLoader(
                    val_subset,
                    batch_size=self.batch_size,
                    shuffle=False,
                    num_workers=self.num_workers,
                    pin_memory=True,
                )

                self.source_train_loaders.append(train_loader)
                self.source_val_loaders.append(val_loader)
                self.source_val_subsets.append(val_subset)
                self.source_domain_names.append(f"{domain_group}/{real_domain}")

                print(
                    f"Source real domain: {domain_group}/{real_domain:8s} "
                    f"| train={len(train_subset)} "
                    f"| val={len(val_subset)} "
                    f"| batches/epoch={len(train_loader)}"
                )

                domain_counter += 1

        if len(self.source_train_loaders) == 0:
            raise ValueError("No source loaders were created.")

        self.source_val_dataset = ConcatDataset(self.source_val_subsets)
        self.source_val_loader = DataLoader(
            self.source_val_dataset,
            batch_size=self.batch_size * len(self.source_train_loaders),
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

        # Build target loader overall + one loader per real target domain.
        self.target_dataset = NICOPPDataset(
            root_path=self.root_path,
            domain_groups=self.target_domain_groups,
            splits=target_splits,
            class_to_idx=self.class_to_idx,
            transform=target_transform,
        )

        if len(self.target_dataset) == 0:
            raise ValueError("Target dataset is empty. Check target_domain and folder structure.")

        self.target_loader = DataLoader(
            self.target_dataset,
            batch_size=self.batch_size * len(self.source_train_loaders),
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

        self.target_real_loaders = []
        self.target_real_domain_names = []

        for target_group in self.target_domain_groups:
            target_real_domains = self._get_real_domains(target_group, target_splits)

            for real_domain in target_real_domains:
                target_real_dataset = NICOPPDataset(
                    root_path=self.root_path,
                    domain_groups=[target_group],
                    splits=target_splits,
                    real_domains=[real_domain],
                    class_to_idx=self.class_to_idx,
                    transform=target_transform,
                )

                if len(target_real_dataset) == 0:
                    continue

                target_real_loader = DataLoader(
                    target_real_dataset,
                    batch_size=self.batch_size * len(self.source_train_loaders),
                    shuffle=False,
                    num_workers=self.num_workers,
                    pin_memory=True,
                )

                self.target_real_loaders.append(target_real_loader)
                self.target_real_domain_names.append(f"{target_group}/{real_domain}")

        print(f"Target samples: {len(self.target_dataset)}")
        print(f"Target real domains: {self.target_real_domain_names}")
        print(f"Batch size per source domain: {self.batch_size}")
        print(f"Effective train batch size: {self.batch_size * len(self.source_train_loaders)}")

        self.optimizer = Adam(
            self.model.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )

        self.criterion = nn.CrossEntropyLoss()

        self.steps_per_epoch = min(len(loader) for loader in self.source_train_loaders)
        if self.steps_per_epoch <= 0:
            raise ValueError("At least one source train loader has zero batches. Try reducing batch_size.")

        self.max_epochs = math.ceil(self.max_iters / self.steps_per_epoch)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=self.max_epochs)

        print(f"Steps per epoch: {self.steps_per_epoch}")
        print(f"Max epochs for scheduler: {self.max_epochs}")

        if self.use_swad:
            print(
                "Using DomainBed-v2 SWAD LossValley "
                f"(n_converge={self.swad_n_converge}, "
                f"n_tolerance={self.swad_n_tolerance}, "
                f"tolerance_ratio={self.swad_tolerance_ratio})"
            )

    def _get_real_domains(self, domain_group, splits):
        real_domains = set()

        for split in splits:
            split_path = Path(self.root_path) / domain_group / split
            if not split_path.exists():
                continue

            for real_domain_dir in split_path.iterdir():
                if real_domain_dir.is_dir():
                    real_domains.add(real_domain_dir.name)

        return sorted(real_domains)

    def get_base_model(self):
        if isinstance(self.model, nn.DataParallel):
            return self.model.module
        return self.model

    def _clone_base_state_dict_cpu(self):
        return {
            k: v.detach().cpu().clone()
            for k, v in self.get_base_model().state_dict().items()
        }

    def evaluate(self, loader, model=None):
        model = self.model if model is None else model
        model.eval()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        with torch.no_grad():
            for images, labels in loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True)

                with autocast("cuda", enabled=self.use_amp):
                    logits = model(images)
                    loss = self.criterion(logits, labels)

                preds = logits.argmax(dim=1)
                batch_size = labels.size(0)

                total_loss += loss.item() * batch_size
                total_correct += (preds == labels).sum().item()
                total_samples += batch_size

        avg_loss = total_loss / total_samples
        avg_acc = total_correct / total_samples

        return {
            "loss": avg_loss,
            "loss_sum": total_loss,
            "acc": avg_acc,
            "total_correct": total_correct,
            "total": total_samples,
        }

    def _evaluate_source_validation(self):
        metrics = self.evaluate(self.source_val_loader)

        per_domain = {}
        for name, loader in zip(self.source_domain_names, self.source_val_loaders):
            per_domain[name] = self.evaluate(loader)

        metrics["per_domain"] = per_domain
        return metrics

    def _next_source_minibatch(self, source_iters):
        images_list = []
        labels_list = []

        for i, loader in enumerate(self.source_train_loaders):
            try:
                images, labels = next(source_iters[i])
            except StopIteration:
                source_iters[i] = iter(loader)
                images, labels = next(source_iters[i])

            images_list.append(images)
            labels_list.append(labels)

        images = torch.cat(images_list, dim=0)
        labels = torch.cat(labels_list, dim=0)
        return images, labels

    def _update_bn_statistics(self, model, source_iters, n_steps=500):
        """
        DomainBed-v2 refreshes BN statistics for SWAD when training from scratch.
        This is important because their AveragedModel averages parameters only.
        """
        momenta = {}
        for module in model.modules():
            if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
                module.running_mean = torch.zeros_like(module.running_mean)
                module.running_var = torch.ones_like(module.running_var)
                momenta[module] = module.momentum

        if len(momenta) == 0:
            return

        was_training = model.training
        model.train()

        for bn_module in momenta.keys():
            bn_module.momentum = None
            bn_module.num_batches_tracked *= 0

        print(f"Updating SWAD BatchNorm statistics for {n_steps} steps...")

        with torch.no_grad():
            for _ in tqdm(range(n_steps), desc="SWAD BN", leave=False):
                images, _ = self._next_source_minibatch(source_iters)
                images = images.to(self.device, non_blocking=True)

                with autocast("cuda", enabled=self.use_amp):
                    model(images)

        for bn_module in momenta.keys():
            bn_module.momentum = momenta[bn_module]

        model.train(was_training)

    def train(self, eval_iters=600, log_iters=20, save_path=None):
        self.model.train()

        print(f"Training with {self.max_iters} iterations")
        print(f"Target group(s): {self.target_domain_groups}")
        print(f"Source real domains: {self.source_domain_names}")
        print(f"eval_iters acts like DomainBed-v2 checkpoint_step_freq: {eval_iters}")

        source_iters = [iter(loader) for loader in self.source_train_loaders]

        best_val_acc = -1.0
        best_state_dict = None
        history = []

        running_loss = 0.0
        running_correct = 0
        running_total = 0

        if self.use_swad:
            self.swad = LossValley(
                n_converge=self.swad_n_converge,
                n_tolerance=self.swad_n_tolerance,
                tolerance_ratio=self.swad_tolerance_ratio,
            )
            self.swad_segment_swa = AveragedModel(self.get_base_model(), device=self.device)

        pbar = tqdm(range(1, self.max_iters + 1), total=self.max_iters)

        for step in pbar:
            images, labels = self._next_source_minibatch(source_iters)

            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            with autocast("cuda", enabled=self.use_amp):
                logits = self.model(images)
                loss = self.criterion(logits, labels)

            self.optimizer.zero_grad(set_to_none=True)
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            # DomainBed-v2 dense segment SWA: update after every optimizer step.
            if self.use_swad:
                self.swad_segment_swa.update_parameters(self.get_base_model(), step=step)

            if step % self.steps_per_epoch == 0:
                self.scheduler.step()

            preds = logits.argmax(dim=1)
            batch_size = labels.size(0)

            running_loss += loss.item() * batch_size
            running_correct += (preds == labels).sum().item()
            running_total += batch_size

            if step % log_iters == 0:
                train_loss = running_loss / running_total
                train_acc = running_correct / running_total

                pbar.set_postfix({
                    "train_loss": f"{train_loss:.4f}",
                    "train_acc": f"{train_acc:.4f}",
                    "lr": f"{self.optimizer.param_groups[0]['lr']:.6f}",
                })

                running_loss = 0.0
                running_correct = 0
                running_total = 0

            if step % eval_iters == 0 or step == self.max_iters:
                source_val_metrics = self._evaluate_source_validation()
                source_val_loss = source_val_metrics["loss"]
                source_val_acc = source_val_metrics["acc"]

                swad_status = "off"

                if self.use_swad:
                    self.swad.update_and_evaluate(
                        self.swad_segment_swa,
                        source_val_acc,
                        source_val_loss,
                    )

                    if self.swad.dead_valley:
                        swad_status = "dead_valley"
                    elif self.swad.is_converged:
                        swad_status = "converged"
                    else:
                        swad_status = "waiting"

                record = {
                    "iter": step,
                    "source_val_loss": source_val_loss,
                    "source_val_acc": source_val_acc,
                    "lr": self.optimizer.param_groups[0]["lr"],
                    "swad_status": swad_status,
                }

                if self.use_swad:
                    record.update({
                        "swad_converged": self.swad.is_converged,
                        "swad_dead_valley": self.swad.dead_valley,
                        "swad_threshold": self.swad.threshold,
                    })

                history.append(record)

                print(
                    f"\nIter [{step}/{self.max_iters}] "
                    f"| Source Val Loss: {source_val_loss:.4f} "
                    f"| Source Val Acc: {source_val_acc:.4f} "
                    f"| LR: {self.optimizer.param_groups[0]['lr']:.6f} "
                    f"| SWAD: {swad_status}"
                )

                # Best IID validation checkpoint is still kept for ERM fallback/comparison.
                if source_val_acc > best_val_acc:
                    best_val_acc = source_val_acc
                    best_state_dict = self._clone_base_state_dict_cpu()

                # DomainBed-v2 resets segment SWA after every checkpoint validation.
                if self.use_swad:
                    if self.swad.dead_valley and self.swad_early_stop:
                        print("SWAD valley is dead -> early stop")
                        break

                    self.swad_segment_swa = AveragedModel(self.get_base_model(), device=self.device)

                self.model.train()

        self.best_val_acc = best_val_acc
        self.best_state_dict = best_state_dict
        self.history = history

        if self.use_swad:
            print("\nFinalizing SWAD model...")
            self.swad_final_model = self.swad.get_final_model(device=self.device)

            if self.swad_update_bn and (not self.pretrained):
                self._update_bn_statistics(
                    self.swad_final_model,
                    source_iters=source_iters,
                    n_steps=self.swad_bn_steps,
                )

            self.swad_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in self.swad_final_model.module.state_dict().items()
            }

            print(
                f"SWAD range: [{self.swad_final_model.start_step}-{self.swad_final_model.end_step}] "
                f"N={int(self.swad_final_model.n_averaged.item())}"
            )

            if save_path is not None:
                torch.save({
                    "algorithm": "DomainBed-v2-style SWAD LossValley",
                    "model_state_dict": self.swad_state_dict,
                    "best_erm_state_dict": self.best_state_dict,
                    "best_val_acc": self.best_val_acc,
                    "swad_start_step": self.swad_final_model.start_step,
                    "swad_end_step": self.swad_final_model.end_step,
                    "swad_n_averaged": int(self.swad_final_model.n_averaged.item()),
                    "swad_n_converge": self.swad_n_converge,
                    "swad_n_tolerance": self.swad_n_tolerance,
                    "swad_tolerance_ratio": self.swad_tolerance_ratio,
                    "swad_threshold": self.swad.threshold,
                    "class_to_idx": self.class_to_idx,
                    "source_domain_names": self.source_domain_names,
                    "target_domain": self.target_domain,
                    "target_real_domain_names": self.target_real_domain_names,
                    "lr": self.lr,
                    "weight_decay": self.weight_decay,
                    "batch_size_per_domain": self.batch_size,
                    "history": self.history,
                }, save_path)
                print(f"Saved SWAD checkpoint to: {save_path}")

        elif save_path is not None and self.best_state_dict is not None:
            torch.save({
                "algorithm": "ERM best IID validation checkpoint",
                "model_state_dict": self.best_state_dict,
                "best_val_acc": self.best_val_acc,
                "class_to_idx": self.class_to_idx,
                "source_domain_names": self.source_domain_names,
                "target_domain": self.target_domain,
                "lr": self.lr,
                "weight_decay": self.weight_decay,
                "batch_size_per_domain": self.batch_size,
                "history": self.history,
            }, save_path)

        print(f"\nBest Source Val Acc: {best_val_acc:.4f}")
        return history

    def evaluate_target(self, use_swad=True, by_real_domain=True):
        if use_swad and self.swad_state_dict is not None:
            print("Loading SWAD averaged weights for target evaluation")
            self.get_base_model().load_state_dict(self.swad_state_dict)
        elif self.best_state_dict is not None:
            print("Loading best ERM/IID-validation weights for target evaluation")
            self.get_base_model().load_state_dict(self.best_state_dict)
        else:
            print("No saved weights found; evaluating current model state")

        overall_metrics = self.evaluate(self.target_loader)

        print("Target Evaluation Overall")
        print(f"Target Loss: {overall_metrics['loss']:.4f}")
        print(f"Target Acc: {overall_metrics['acc']:.4f}")
        print(f"Correct: {overall_metrics['total_correct']} / {overall_metrics['total']}")

        result = {
            "overall": overall_metrics,
            "per_domain": {},
        }

        if by_real_domain and len(self.target_real_loaders) > 0:
            print("\nTarget Evaluation By Real Domain")
            for name, loader in zip(self.target_real_domain_names, self.target_real_loaders):
                metrics = self.evaluate(loader)
                result["per_domain"][name] = metrics
                print(
                    f"{name:30s} "
                    f"| Loss: {metrics['loss']:.4f} "
                    f"| Acc: {metrics['acc']:.4f} "
                    f"| Correct: {metrics['total_correct']} / {metrics['total']}"
                )

        return result


In [ ]:
source_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.3),
    transforms.RandomGrayscale(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

target_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
trainer = Trainer(
    model_path="mobilenetv4_conv_small.e2400_r224_in1k",
    num_classes=60,
    pretrained=False,
    target_domain="autumn_rock",
    source_transform=source_transform,
    target_transform=target_transform,
    batch_size=32,
    lr=1e-4,
    weight_decay=1e-4,
    max_iters=10000,      # DomainBed-v2 uses 10000 steps for NICO/NICO++
    val_ratio=0.2,
    source_splits=("train",),           # source train is internally split into train/val
    target_splits=("test",),            # target is held out for final evaluation
    num_workers=1,
    use_amp=True,
    use_data_parallel=False,
    use_swad=True,
    swad_n_converge=3,
    swad_n_tolerance=6,
    swad_tolerance_ratio=0.3,
    swad_update_bn=True,
    swad_bn_steps=500,
    swad_early_stop=True,
)


In [ ]:
history = trainer.train(
    eval_iters=600,      # DomainBed-v2 checkpoint_step_freq for NICO/NICO++
    log_iters=20,        # DomainBed-v2 stepval_freq default
    save_path="/kaggle/working/best_swad_domainbedv2.pth"
)


In [ ]:
NICO_PP_TARGET_GROUPS = ["autumn_rock", "dim_grass", "outdoor_water"]
NICO_PP_PAPER_DOMAIN_ORDER = ["autumn", "rock", "dim", "grass", "outdoor", "water"]


def paper_metric_row(results_by_group, algorithm_name=None):
    """
    Return NICO++ paper-style columns: each real target-domain accuracy in percent
    plus the macro Avg across available target domains.

    Pass either one evaluate_target() result or a dict such as:
        {"autumn_rock": autumn_rock_results, "dim_grass": dim_grass_results, ...}
    """
    if isinstance(results_by_group, dict) and (
        "domain_metrics" in results_by_group or "per_domain" in results_by_group
    ):
        grouped_results = {results_by_group.get("target_group", "target"): results_by_group}
    else:
        grouped_results = results_by_group

    accs = {}
    for _, result in grouped_results.items():
        if isinstance(result, dict) and "target_results" in result:
            result = result["target_results"]

        domain_metrics = result.get("domain_metrics", result.get("per_domain", {}))
        for domain, metrics in domain_metrics.items():
            real_domain = str(domain).split("/")[-1]
            acc = float(metrics["acc"])
            accs[real_domain] = acc * 100.0 if acc <= 1.0 else acc

    row = {domain: accs.get(domain, np.nan) for domain in NICO_PP_PAPER_DOMAIN_ORDER}
    values = list(row.values())
    row["Avg"] = float(np.nanmean(values)) if np.isfinite(values).any() else np.nan

    if algorithm_name is not None:
        row = {"Algorithm": algorithm_name, **row}

    return pd.DataFrame([row])


In [ ]:
target_results = trainer.evaluate_target(use_swad=True, by_real_domain=True)

paper_metrics = paper_metric_row(target_results, algorithm_name="SWAD")
paper_metrics
